In [1]:
!pip install datasketch spacy langdetect keybert sentence-transformers
!python -m spacy download en_core_web_sm



[notice] A new release of pip is available: 24.0 -> 25.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


     ---------------------------------------- 0.0/12.8 MB ? eta -:--:--
     ---------------------------------------- 0.0/12.8 MB ? eta -:--:--
     ---------------------------------------- 0.0/12.8 MB ? eta -:--:--
     ---------------------------------------- 0.0/12.8 MB ? eta -:--:--
     ---------------------------------------- 0.0/12.8 MB ? eta -:--:--
     ---------------------------------------- 0.0/12.8 MB ? eta -:--:--
     ---------------------------------------- 0.0/12.8 MB ? eta -:--:--
     ---------------------------------------- 0.0/12.8 MB ? eta -:--:--
     ---------------------------------------- 0.0/12.8 MB ? eta -:--:--
     ---------------------------------------- 0.0/12.8 MB ? eta -:--:--
     ---------------------------------------- 0.0/12.8 MB ? eta -:--:--
     ---------------------------------------- 0.0/12.8 MB ? eta -:--:--
     ---------------------------------------- 0.0/12.8 MB ? eta -:--:--
     ---------------------------------------- 0.0/12.8 MB ? eta 


[notice] A new release of pip is available: 24.0 -> 25.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
import pandas as pd

df = pd.read_csv("merged_news.csv")  # uses comma by default

In [5]:
print(df.columns.tolist())


['timestamp', 'title', 'summary', 'link', 'source']


In [6]:
# Combine 'title' and 'summary' into one text column
df["text"] = df["title"].fillna('') + ". " + df["summary"].fillna('')


In [7]:
print(df.head)

<bound method NDFrame.head of                          timestamp  \
0       2025-05-22T11:29:05.503214   
1       2025-05-22T11:29:05.503214   
2       2025-05-22T11:29:05.503214   
3       2025-05-22T11:29:05.503214   
4       2025-05-22T11:29:05.503214   
..                             ...   
931  Wed, 28 May 2025 22:42:05 GMT   
932  Thu, 29 May 2025 08:14:30 GMT   
933  Thu, 29 May 2025 07:45:48 GMT   
934  Thu, 29 May 2025 08:56:26 GMT   
935  Thu, 29 May 2025 12:27:48 GMT   

                                                 title  \
0    LIVETwo Israeli embassy staff killed and suspe...   
1    US Jewish museum shooting suspect was mistaken...   
2    LIVE'Multiple fatalities' on private plane tha...   
3    LIVEUS House passes Trump tax and spending meg...   
4    Watch: Deep inside a Norwegian mountain, Nato ...   
..                                                 ...   
931  Hailey Bieber's make-up brand Rhode sold in $1...   
932  UK turns to AI and drones for new battlefiel

# Convert timestamp to simplified datetime

In [9]:
df["datetime"] = pd.to_datetime(df["timestamp"], errors="coerce", utc=True)


In [10]:
df = df[df["datetime"].notna()]  # Remove rows with NaT

# Format the datetime as string: YYYY-MM-DD HH:MM
df["datetime"] = df["datetime"].dt.strftime("%Y-%m-%d %H:%M")


# Clean title and summary (remove "LIVE", trim spaces, etc.)

In [11]:
import re

def clean_text(text):
    if pd.isnull(text):
        return ""
    text = re.sub(r"\bLIVE\b", "", text, flags=re.IGNORECASE)  # remove "LIVE"
    text = re.sub(r"http\S+", "", text)                        # remove URLs
    text = re.sub(r"[^\w\s.,!?-]", "", text)                   # remove weird chars
    return text.strip()

df["title_clean"] = df["title"].apply(clean_text)
df["summary_clean"] = df["summary"].fillna("").apply(clean_text)


# Create text column for deduplication

In [12]:

df["text"] = df["title_clean"] + ". " + df["summary_clean"]
df["text"] = df["text"].fillna("")  # Just in case


# Deduplication using MinHash

In [13]:
from datasketch import MinHash, MinHashLSH

def get_minhash(text):
    m = MinHash(num_perm=128)
    for word in text.split():
        m.update(word.encode('utf8'))
    return m

lsh = MinHashLSH(threshold=0.85, num_perm=128)
unique_indices = []

for idx, row in df.iterrows():
    m = get_minhash(row["text"])
    if not any(lsh.query(m)):
        lsh.insert(f"text_{idx}", m)
        unique_indices.append(idx)

df = df.loc[unique_indices].reset_index(drop=True)



# Named Entity Recognition (NER)

In [14]:
!python -m pip install --upgrade "optree>=0.13.0"




[notice] A new release of pip is available: 24.0 -> 25.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [15]:
!pip install spacy
!python -m spacy download en_core_web_sm



[notice] A new release of pip is available: 24.0 -> 25.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


     ---------------------------------------- 0.0/12.8 MB ? eta -:--:--
     ---------------------------------------- 0.0/12.8 MB ? eta -:--:--
     ---------------------------------------- 0.0/12.8 MB ? eta -:--:--
     ---------------------------------------- 0.0/12.8 MB ? eta -:--:--
     ---------------------------------------- 0.0/12.8 MB ? eta -:--:--
     ---------------------------------------- 0.0/12.8 MB ? eta -:--:--
     ---------------------------------------- 0.0/12.8 MB ? eta -:--:--
     --------------------------------------- 0.0/12.8 MB 130.4 kB/s eta 0:01:38
     --------------------------------------- 0.0/12.8 MB 130.4 kB/s eta 0:01:38
     --------------------------------------- 0.0/12.8 MB 130.4 kB/s eta 0:01:38
     --------------------------------------- 0.0/12.8 MB 130.4 kB/s eta 0:01:38
     --------------------------------------- 0.0/12.8 MB 130.4 kB/s eta 0:01:38
     --------------------------------------- 0.0/12.8 MB 130.4 kB/s eta 0:01:38
     -----------


[notice] A new release of pip is available: 24.0 -> 25.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [17]:
# Step 2: Import spaCy and load the model
import spacy

# Load small English model
nlp = spacy.load("en_core_web_sm")

# Step 3: Define NER function to extract (entity, label) pairs
ALLOWED_ENTITY_TYPES = ["PERSON", "ORG", "GPE", "DATE", "TIME", "MONEY", "PERCENT", "NORP", "EVENT", "LAW", "FAC", "PRODUCT"]  # You can add more

def extract_filtered_entities(text):
    if not isinstance(text, str) or not text.strip():
        return []
    doc = nlp(text)
    return list(set((ent.text, ent.label_) for ent in doc.ents if ent.label_ in ALLOWED_ENTITY_TYPES))

# Step 4: Apply the function to the 'text' column
df["entities"] = df["text"].apply(extract_filtered_entities)

# Step 5: Preview result
df[["text", "entities"]].head()



,text,entities
0,LIVETwo Israeli embassy staff killed and suspe...,"[(Jewish, NORP), (Israeli, NORP), (Yaron, ORG)..."
1,US Jewish museum shooting suspect was mistaken...,"[(US, GPE), (agoWorld, ORG), (Jewish, NORP)]"
2,Multiple fatalities on private plane that cras...,"[(sayCalifornia, GPE), (San Diego, GPE)]"
3,LIVEUS House passes Trump tax and spending meg...,"[(Senate, ORG), (LIVEUS House, ORG), (Trump, O..."
4,"Watch Deep inside a Norwegian mountain, Nato a...","[(Katya Adler, PERSON), (Nato, ORG), (Norwegia..."


# Separate Entity Text and Label

In [18]:
df["entity_names"] = df["entities"].apply(lambda ents: [e[0] for e in ents])
df["entity_types"] = df["entities"].apply(lambda ents: [e[1] for e in ents])


# Keyphrase Extraction (TF-IDF)

In [19]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Vectorize using unigrams + bigrams, remove common stop words
vectorizer = TfidfVectorizer(ngram_range=(1, 2), stop_words="english")
tfidf_matrix = vectorizer.fit_transform(df["text"])
feature_names = vectorizer.get_feature_names_out()

# Extract top N keywords per row
def get_top_keywords(row_index, top_n=5):
    row = tfidf_matrix[row_index].toarray().flatten()
    top_indices = row.argsort()[-top_n:][::-1]  # top 5 in descending order
    return [feature_names[i] for i in top_indices]

# Apply to all rows
df["keyphrases"] = [get_top_keywords(i) for i in range(len(df))]



# Language Detection

In [20]:
from langdetect import detect

def detect_lang(text):
    try:
        return detect(text)
    except:
        return "unknown"

df["language"] = df["text"].apply(detect_lang)


In [21]:
df["language"].value_counts()


en    896
es      3
lt      2
de      2
so      1
af      1
ro      1
id      1
ja      1
ru      1
nl      1
pl      1
tl      1
sv      1
ca      1
fi      1
no      1
da      1
Name: language, dtype: int64

# Translation (Only Non-English Rows)

In [22]:
!pip install sentencepiece



[notice] A new release of pip is available: 24.0 -> 25.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [23]:
!pip install transformers sentencepiece tqdm



[notice] A new release of pip is available: 24.0 -> 25.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [24]:
from transformers import MarianMTModel, MarianTokenizer
from tqdm import tqdm

# Enable progress bar in DataFrame apply
tqdm.pandas()

# Load model and tokenizer
model_name = "Helsinki-NLP/opus-mt-mul-en"
tokenizer = MarianTokenizer.from_pretrained(model_name)
model = MarianMTModel.from_pretrained(model_name)


source.spm:   0%|          | 0.00/707k [00:00<?, ?B/s]

C:\Users\ASUS\anaconda3\Lib\site-packages\huggingface_hub\file_download.py:144: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\ASUS\.cache\huggingface\hub\models--Helsinki-NLP--opus-mt-mul-en. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


target.spm:   0%|          | 0.00/791k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.42M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.40k [00:00<?, ?B/s]

C:\Users\ASUS\anaconda3\Lib\site-packages\transformers\models\marian\tokenization_marian.py:175: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")


pytorch_model.bin:   0%|          | 0.00/310M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/293 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/310M [00:00<?, ?B/s]

In [25]:
def translate_text(text):
    try:
        tokens = tokenizer([text], return_tensors="pt", truncation=True, padding=True)
        translated = model.generate(**tokens)
        return tokenizer.decode(translated[0], skip_special_tokens=True)
    except Exception as e:
        print(f"Translation failed: {e}")
        return text  # Return original text on error


In [26]:
df["translated_text"] = df.progress_apply(
    lambda x: translate_text(x["text"]) if x["language"] != "en" else x["text"],
    axis=1
)


100%|████████████████████████████████████████████████████████████████████████████████| 917/917 [00:17<00:00, 52.94it/s]


In [27]:
df.to_csv("preprocessed_news.csv", index=False)
print("✅ Preprocessed data saved to 'preprocessed_news.csv'")


✅ Preprocessed data saved to 'preprocessed_news.csv'
